# Grupo 4


*   Camila Contreras

*   Christian Perez
*   Javier Uribe





## Tarea: Modelo de Regresión Lineal para Presión Sistólica (PAS)

### Objetivo:
Construir un modelo de **regresión lineal múltiple** para predecir la **presión arterial sistólica** (`m2p11a_PAS`) utilizando variables clínicas y el nivel de riesgo cardiovascular (`RCV_CHILENO_RECODIFICADO`).

---

### Variables a utilizar:

- `Edad` — Edad  
- `Sexo` — Sexo *(categórica)*  
- `Glucosa` — Glicemia  
- `Colesterol_Total` — Colesterol total  
- `Colesterol_HDL` — Colesterol HDL  
- `IMC` — Índice de masa corporal  
- `RCV_CHILENO_RECODIFICADO` — Nivel de riesgo cardiovascular (categórica con tres niveles)  
- `m2p11a_PAS` — **Presión arterial sistólica (variable objetivo)**  

---

### Instrucciones:

1. Usa **todas las variables listadas** como predictoras de `m2p11a_PAS`, excepto esta última que será la variable dependiente.

2. **Calcula el porcentaje de datos faltantes por variable** y luego elimina las filas con datos incompletos usando `.dropna()`.

3. **Convierte las variables categóricas** `Sexo` y `RCV_CHILENO_RECODIFICADO` en variables dummy con `pd.get_dummies()` y `drop_first=True`.

4. Ajusta un modelo de **regresión lineal múltiple** con `statsmodels.OLS`.

5. **Interpreta los resultados**:
   - ¿Qué variables tienen un efecto estadísticamente significativo?
   - ¿Cuál es el valor de R²?
   - ¿Qué implicancias tienen los coeficientes estimados?

6. ¿Qué ocurre si agregamos la variable `m2p11a_PAD` (presión arterial diastólica) como una variable predictora adicional en el modelo?

- ¿Cambia la significancia de las otras variables?
- ¿Mejora el ajuste del modelo (R²)?
- ¿Cómo se interpreta su coeficiente?

Agrega `m2p11a_PAD` a la lista de variables y vuelve a ajustar el modelo para responder estas preguntas.


In [1]:
#Codigos previos para desarrollar la tarea

In [2]:
# Descargar ENS
!wget https://github.com/iHealthInstitute/Talleres_Diplomado_iHealth/raw/refs/heads/main/data/ENS2016-2017.sav

--2025-11-29 03:10:26--  https://github.com/iHealthInstitute/Talleres_Diplomado_iHealth/raw/refs/heads/main/data/ENS2016-2017.sav
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/iHealthInstitute/Talleres_Diplomado_iHealth/refs/heads/main/data/ENS2016-2017.sav [following]
--2025-11-29 03:10:26--  https://raw.githubusercontent.com/iHealthInstitute/Talleres_Diplomado_iHealth/refs/heads/main/data/ENS2016-2017.sav
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 34441286 (33M) [application/octet-stream]
Saving to: ‘ENS2016-2017.sav’

ENS2016-2017.sav    100%[===================>]  32.85M  --.-KB/s    

In [3]:

# Instalar librerias para leer archivos SPSS
!pip install pyreadstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 17.9 MB/s eta 0:00:00


In [4]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
import statsmodels.api as sm

In [5]:
# cargar los datos de la encuesta nacional de salud.
#Utilizando la variable ens para almacenar el DataFrame que contiene los datos de la encuesta (ENS2016-2017.sav).

ens = pd.read_spss('ENS2016-2017.sav')
print(ens.head())
print(ens.shape)

ens.info()

   IdEncuesta  FechaInicioF1               Region    Comuna    Zona  \
0     20006.0   1.369894e+10  XIII. Metropolitana  Santiago  URBANA   
1     20008.0   1.369964e+10  XIII. Metropolitana  Santiago  URBANA   
2     20011.0   1.369955e+10  XIII. Metropolitana  Santiago  URBANA   
3     20012.0   1.369903e+10  XIII. Metropolitana  Santiago  URBANA   
4     20013.0   1.369902e+10  XIII. Metropolitana  Santiago  URBANA   

   IdSegmento  IdPersona_1      Ident7  Edad Edad_Codificada  ...  \
0  13101101.0        241.0  1977-12-18  38.0         25 - 44  ...   
1  13101101.0        197.0  1991-10-23  25.0         25 - 44  ...   
2  13101102.0        321.0  1996-05-31  20.0           15-24  ...   
3  13101102.0        245.0  1931-04-14  85.0             65+  ...   
4  13101102.0        242.0  1975-06-24  41.0         25 - 44  ...   

  fg_CKDschwartz_diminuido_60 fg_CKDschwartz_diminuido_30 Fechaini_F1  \
0                         NaN                         NaN  2016-11-19   
1           

In [6]:
#01Cargar librerías y seleccionar variables

import pandas as pd
import statsmodels.api as sm

# Cargar el DataFrame (que se llama ens)
# ens = pd.read_csv("archivo.csv")

variables = [
    "Edad",
    "Sexo",
    "Glucosa",
    "Colesterol_Total",
    "Colesterol_HDL",
    "IMC",
    "RCV_CHILENO_RECODIFICADO",
    "m2p11a_PAS"
]

ens_modelo = ens[variables].copy()


In [7]:
#2. Calcular porcentaje de datos faltantes

faltantes = ens_modelo.isnull().mean() * 100
print("Porcentaje de datos faltantes por variable:")
print(faltantes)


Porcentaje de datos faltantes por variable:
Edad                         0.000000
Sexo                         0.000000
Glucosa                     17.904701
Colesterol_Total            40.397882
Colesterol_HDL              40.397882
IMC                         12.032729
RCV_CHILENO_RECODIFICADO    44.200225
m2p11a_PAS                  11.503289
dtype: float64


In [8]:
#3. Eliminar filas con valores faltantes

ens_modelo = ens_modelo.dropna()
print("Tamaño del dataset luego de eliminar filas incompletas:", ens_modelo.shape)


Tamaño del dataset luego de eliminar filas incompletas: (3435, 8)


In [9]:
#4. Convertir variables categóricas en variables dummy

df_dummies = pd.get_dummies(
    ens_modelo,
    columns=["Sexo", "RCV_CHILENO_RECODIFICADO"],
    drop_first=True
)

df_dummies.head()


,Edad,Glucosa,Colesterol_Total,Colesterol_HDL,IMC,m2p11a_PAS,Sexo_MUJER,RCV_CHILENO_RECODIFICADO_10+:10+: Alto,RCV_CHILENO_RECODIFICADO_5-9:Moderado
1,25.0,79.0,159.0,46.0,23.198208,93.667,True,False,False
2,20.0,78.0,172.0,52.0,26.203983,100.333,True,False,False
3,85.0,101.0,295.0,26.0,23.732346,216.667,True,True,False
6,53.0,95.0,217.0,44.0,32.290249,101.667,True,True,False
7,75.0,104.0,199.0,56.0,30.939320,131.667,True,True,False


In [10]:
# 1. Seleccionar solo columnas booleanas
columnas_booleanas = df_dummies.select_dtypes(include=['bool']).columns
# 2. Transformarlas a 0 y 1
df_dummies[columnas_booleanas] = df_dummies[columnas_booleanas].astype(int)

# # Convertir 'm2p11a_PAS' a tipo numérico
df_dummies["m2p11a_PAS"] = pd.to_numeric(
    df_dummies["m2p11a_PAS"],
    errors='coerce'
).astype(float)

df_dummies.head()
df_dummies.info()



<class 'pandas.core.frame.DataFrame'>
Index: 3435 entries, 1 to 6232
Data columns (total 9 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Edad                                    3435 non-null   float64
 1   Glucosa                                 3435 non-null   float64
 2   Colesterol_Total                        3435 non-null   float64
 3   Colesterol_HDL                          3435 non-null   float64
 4   IMC                                     3435 non-null   float64
 5   m2p11a_PAS                              3435 non-null   float64
 6   Sexo_MUJER                              3435 non-null   int64  
 7   RCV_CHILENO_RECODIFICADO_10+:10+: Alto  3435 non-null   int64  
 8   RCV_CHILENO_RECODIFICADO_5-9:Moderado   3435 non-null   int64  
dtypes: float64(6), int64(3)
memory usage: 268.4 KB


In [11]:
df_dummies.corr()

,Edad,Glucosa,Colesterol_Total,Colesterol_HDL,IMC,m2p11a_PAS,Sexo_MUJER,RCV_CHILENO_RECODIFICADO_10+:10+: Alto,RCV_CHILENO_RECODIFICADO_5-9:Moderado
Edad,1.000000,0.232268,0.197792,0.064595,0.105707,0.565780,0.049518,0.479614,0.005595
Glucosa,0.232268,1.000000,0.104699,-0.110904,0.170823,0.230622,-0.001094,0.424994,-0.086181
Colesterol_Total,0.197792,0.104699,1.000000,0.147999,0.081916,0.162499,0.062279,0.085009,0.095569
Colesterol_HDL,0.064595,-0.110904,0.147999,1.000000,-0.257246,0.006983,0.214964,-0.092395,-0.277789
IMC,0.105707,0.170823,0.081916,-0.257246,1.000000,0.165782,0.113232,0.172060,0.205653
m2p11a_PAS,0.565780,0.230622,0.162499,0.006983,0.165782,1.000000,-0.135479,0.365916,0.073865
Sexo_MUJER,0.049518,-0.001094,0.062279,0.214964,0.113232,-0.135479,1.000000,0.027803,-0.038753
RCV_CHILENO_RECODIFICADO_10+:10+: Alto,0.479614,0.424994,0.085009,-0.092395,0.172060,0.365916,0.027803,1.000000,-0.372872
RCV_CHILENO_RECODIFICADO_5-9:Moderado,0.005595,-0.086181,0.095569,-0.277789,0.205653,0.073865,-0.038753,-0.372872,1.000000


In [12]:
#5Ajustar modelo de regresion lineal multiple PAS como dependiente)

X = df_dummies.drop("m2p11a_PAS", axis=1)
y = df_dummies["m2p11a_PAS"]

# Agregar constante
X = sm.add_constant(X)

modelo1 = sm.OLS(y, X).fit()
print(modelo1.summary())

# Mostrar ecuación del modelo
coeficientes = modelo1.params

print("\nEcuación del modelo:\n")
equacion = f"m2p11a_PAS = {coeficientes['const']:.4f}"

for nombre, valor in coeficientes.items():
    if nombre != "const":
        equacion += f" + ({valor:.4f})*{nombre}"

print(equacion)



                            OLS Regression Results                            
Dep. Variable:             m2p11a_PAS   R-squared:                       0.391
Model:                            OLS   Adj. R-squared:                  0.390
Method:                 Least Squares   F-statistic:                     275.1
Date:                Sat, 29 Nov 2025   Prob (F-statistic):               0.00
Time:                        03:13:40   Log-Likelihood:                -14526.
No. Observations:                3435   AIC:                         2.907e+04
Df Residuals:                    3426   BIC:                         2.913e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                                             coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------

#Interpreta los resultados:

¿Qué variables tienen un efecto estadísticamente significativo?

Todas son estadísticamente significativas en explicar variación en PAS(si p < 0.05) :

✔ Edad (p < 0.001)

✔ Glucosa (p < 0.001)

✔ Colesterol Total (p = 0.045)

✔ Colesterol HDL (p < 0.001)

✔ IMC (p < 0.001)

✔ Sexo mujer (p < 0.001)

✔ RCV Alto (p < 0.001)

✔ RCV Moderado (p < 0.001)

¿Cuál es el valor de R²?

Para este modelo R² = 0.391 (R² ajustado = 0.390) Esto significa que el modelo explica aproximadamente el 39% de la variabilidad de la presión arterial sistólica.

¿Qué implicancias tienen los coeficientes estimados?

Edad (β = 0.509 mmHg por año) Por cada 1 año extra, la PAS aumenta en 0.51 mmHg. En 20 años, serían ~10 mmHg más.

Glucosa (β = 0.032 mmHg por mg/dL) Un aumento de 10 mg/dL → sube 0.32 mmHg. Efecto pequeño, pero estadísticamente real.

Colesterol total (β = 0.015) Cada 10 mg/dL → +0.15 mmHg Muy pequeño, pero significativo (p = 0.045).

HDL (β = 0.159 mmHg por mg/dL) A mayor HDL, mayor PAS, lo cual es una relación inusual pero puede ocurrir si: hay colinealidad con otras variables, HDL actúa como marcador de mejor estado metabólico, el efecto real es débil.

IMC (β = 0.365 mmHg por punto de IMC) Cada punto extra de IMC → +0.37 mmHg.

Sexo: mujer (β = –8.45 mmHg) Ser mujer predice una presión 8.4 mmHg menor que ser hombre, controlando todo lo demás.

RCV chileno Alto: +7.8 mmHg Moderado: +6.8 mmHg A mayor riesgo cardiovascular estimado → mayor PAS.

DESARROLLO PREGUNTA 6

In [13]:
#Respuesta pregunta 6. Añadir "m2p11a_PAD" a las variables.

ens = pd.read_spss('ENS2016-2017.sav')
variables2 = [
    "Edad",
    "Sexo",
    "Glucosa",
    "Colesterol_Total",
    "Colesterol_HDL",
    "IMC",
    "RCV_CHILENO_RECODIFICADO",
    "m2p11a_PAS",
    "m2p11a_PAD"
]

ens_modelo2 = ens[variables2].copy()
ens_modelo2 = ens_modelo2.dropna()
df_dummies2 = pd.get_dummies(
    ens_modelo2,
    columns=["Sexo", "RCV_CHILENO_RECODIFICADO"],
    drop_first=True
)

# Identificar columnas booleanas
cols_bool = df_dummies2.select_dtypes(include=['bool']).columns

# Convertirlas a int (0 y 1)
df_dummies2[cols_bool] = df_dummies2[cols_bool].astype(int)


# Convertir m2p11a_PAS y m2p11a_PAD a numérico
df_dummies2[["m2p11a_PAS", "m2p11a_PAD"]] = df_dummies2[["m2p11a_PAS", "m2p11a_PAD"]].apply(
    pd.to_numeric, errors='coerce'
).astype(float)

df_dummies2.head()
df_dummies2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3435 entries, 1 to 6232
Data columns (total 10 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   Edad                                    3435 non-null   float64
 1   Glucosa                                 3435 non-null   float64
 2   Colesterol_Total                        3435 non-null   float64
 3   Colesterol_HDL                          3435 non-null   float64
 4   IMC                                     3435 non-null   float64
 5   m2p11a_PAS                              3435 non-null   float64
 6   m2p11a_PAD                              3435 non-null   float64
 7   Sexo_MUJER                              3435 non-null   int64  
 8   RCV_CHILENO_RECODIFICADO_10+:10+: Alto  3435 non-null   int64  
 9   RCV_CHILENO_RECODIFICADO_5-9:Moderado   3435 non-null   int64  
dtypes: float64(7), int64(3)
memory usage: 295.2 KB


In [14]:
#5Ajustar modelo de regresion lineal multiple PAS como dependiente)

X2 = df_dummies2.drop("m2p11a_PAS", axis=1)
y2 = df_dummies2["m2p11a_PAS"]

# Agregar constante
X2 = sm.add_constant(X2)

modelo2 = sm.OLS(y2, X2).fit()
print(modelo2.summary())

# Mostrar ecuación del modelo 2
coeficientes2 = modelo2.params

print("\nEcuación del modelo 2:\n")
ecuacion2 = f"m2p11a_PAS = {coeficientes2['const']:.4f}"

for nombre, valor in coeficientes2.items():
    if nombre != "const":
        ecuacion2 += f" + ({valor:.4f})*{nombre}"

print(ecuacion2)


                            OLS Regression Results                            
Dep. Variable:             m2p11a_PAS   R-squared:                       0.626
Model:                            OLS   Adj. R-squared:                  0.625
Method:                 Least Squares   F-statistic:                     637.7
Date:                Sat, 29 Nov 2025   Prob (F-statistic):               0.00
Time:                        03:16:28   Log-Likelihood:                -13688.
No. Observations:                3435   AIC:                         2.740e+04
Df Residuals:                    3425   BIC:                         2.746e+04
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                                             coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------

#Comparamos las dos ecuaciones

**Ecuación del modelo1**:

m2p11a_PAS = 78.9964 + (0.5091)*Edad + (0.0322)*Glucosa + (0.0150)*Colesterol_Total + (0.1589)*Colesterol_HDL + (0.3654)*IMC + (-8.4450)*Sexo_MUJER + (7.8030)*RCV_CHILENO_RECODIFICADO_10+:10+: Alto + (6.8295)*RCV_CHILENO_RECODIFICADO_5-9:Moderado\

**Ecuación del modelo 2**:

m2p11a_PAS = 22.5891 + (0.4626)*Edad + (0.0221)*Glucosa + (-0.0301)*Colesterol_Total + (0.1053)*Colesterol_HDL + (-0.1021)*IMC + (1.1084)*m2p11a_PAD + (-2.9095)*Sexo_MUJER + (4.7828)*RCV_CHILENO_RECODIFICADO_10+:10+: Alto + (2.0316)*RCV_CHILENO_RECODIFICADO_5-9:Moderado

**Comparación de los Modelos**

Modelo 1 predice m2p11a_PAS usando edad, glucosa, colesterol total, colesterol HDL, IMC y dummies de sexo/RCV.

Modelo 2 usa las mismas variables pero además incluye la presión diastólica (m2p11a_PAD).*

**¿Qué ocurre si agregamos la variable m2p11a_PAD (presión arterial diastólica) como una variable predictora adicional en el modelo?**

La PAD (P.A diastólica) es un predictor fisiológico muy fuerte de la presión sistólica, por lo que modifica las demás variables. Todas siguen siendo significativas, pero las más PAD es la más fuerte asociada a las PAS.


**¿Mejora el ajuste del modelo (R²)?**

Si, al tener un valor de 0.626 (En modelo 1 tiene un valor menor de 0.391).
El modelo 2 explica aproximadamente el 63% de la variabilidad de la presión arterial sistólica. Mejora. con un aumento de casi un 24%.
PAD se convierte en el predictor dominante.

La mayoría de los otros coeficientes disminuyen o cambian de signo.
Explicación más limpia y más “realista” desde el punto de vista clínico.

**¿Cómo se interpreta su coeficiente?**

Las variables determinantes mas fuertes de la P.A Sistólica son PAD (β = 1.1084), RCV Alto (β = +4.78), Moderado (β = +2.03) y Edad (β = 0.4626).

Si las deglosamos por variable comparando M1/M2


*Edad M1: 0.5091/ M2: 0.4626*

  Similar en ambos modelos → la edad mantiene su efecto directo sobre la presión sistólica.


*Glucosa  M1: 0.0322/ M2: 0.0221*

  Baja un poco → parte de la correlación glucosa PAS está mediada o compartida con PAD.

*Colesterol Total  M1: 0.0150 / M2: -0.0301*

  Cambia de signo → cuando agregas PAD, el colesterol total ya no explica PAS de manera independiente.

*Colesterol HDL    M1: 0.1589 / M2: 0.1053*

  Disminuye, pero sigue siendo positivo.

*IMC   M1: 0.3654 / M2: -0.1021*

  Cambia de positivo a levemente negativo.

*Sexo (mujer)    M1: -8.4450 / M2: -2.9095*

El efecto disminuye al controlar por PAD.

*RCV Alto    M1: 7.8030 / M2: 4.78285*

*RCV Moderado  M1: 6.8295  / M2: 2.0316*

  Ambos efectos caen con fuerza.
  Esto indica que PAD absorbe parte del riesgo cardiovascular reflejado en PAS.

**m2p11a_PAD (solo en Modelo 2) M2: 1.1084**


 El coeficiente más alto de todo el modelo.
Por cada aumento de 1 mmHg en presión diastólica, la presión sistólica aumenta en 1.1 mmHg, manteniendo todo lo demás constante.
Este valor explica por qué muchas otras variables pierden fuerza al ser añadida la PAD.

El Modelo 2 es claramente superior porque:

Incluye un predictor fundamental (PAD).

Reduce sesgos por variables omitidas.

Aísla mejor el efecto real de edad, sexo, y RCV sobre PAS.
